# 🚀 KAGGLE MAX-PING ALL-IN-ONE: AUTO-CONVERT + FINE-TUNE YOLOv12-LARGE (BDD100K + nuScenes, GPU T4 × 2)
### ⚡ Cách chạy nhanh nhất (tốn đúng 2 phút chuẩn bị, 100% không xung đột NumPy trên Kaggle Python 3.13!):
1. **Accelerator:** Chọn **`GPU T4 x2`** | **Internet:** Bật **`ON`**.
2. **Add Input** bên cột phải, gắn 3 nguồn:
   - Dataset chứa file **`yolo_best_kaggle_base.pt`** (`51 MB`).
   - **`solesensei/solesensei_bdd100k`** (BDD100K gốc).
   - **`aadimator/nuscenes-mini`** (nuScenes-mini gốc — xử lý trực tiếp bằng **Pure-NumPy**, không cài `nuscenes-devkit` gây lỗi NumPy!).
3. Bấm **Save Version → Save & Run All (Commit)** rồi đi ngủ! Notebook tự động chuyển đổi nhãn chuẩn 3 lớp (`0: Vehicle, 1: Pedestrian, 2: Traffic Light`), train xuyên đêm (`time=10.5h`), đánh giá **TTA**, và xuất gọn toàn bộ `yolo_best.pt` + biểu đồ + `results.csv` vào `/kaggle/working/FINAL_EXPORT/`.

In [ ]:
# ==============================================================================
# BƯỚC 1: CÀI ĐẶT ULTRALYTICS AN TOÀN (KHÔNG ĐỤNG NUMPY/TORCH CỦA KAGGLE)
# ==============================================================================
!pip install -q ultralytics --upgrade-strategy only-if-needed

import os, glob, json, shutil, torch, yaml
import numpy as np
from PIL import Image
import ultralytics
from ultralytics import YOLO

ultralytics.checks()
NUM_GPUS = torch.cuda.device_count()
DEVICES = list(range(NUM_GPUS)) if NUM_GPUS >= 2 else (0 if NUM_GPUS == 1 else 'cpu')
# YOLOv12-Large có lớp Area-Attention tốn VRAM ở imgsz=800 -> 4 ảnh/GPU chống OOM 100% trên T4 15GB
BATCH_SIZE = 8 if NUM_GPUS >= 2 else 4   # 4 ảnh/GPU x 2 GPU T4 = 8 (tích lũy gradient nbs=64)

print(f'✅ Phát hiện {NUM_GPUS} GPU | DEVICES={DEVICES} | BATCH_SIZE={BATCH_SIZE} | NumPy={np.__version__}')

In [ ]:
# ==============================================================================
# BƯỚC 2: TỰ ĐỘNG QUÉT TRỌNG SỐ + AUTO-CONVERT BDD100K & nuScenes SANG 3 LỚP YOLO
# Dùng Symlink ảnh (0 MB đĩa) + Pure-NumPy cho nuScenes (ZERO pip install, ZERO lỗi NumPy!)
# ==============================================================================
INPUT_ROOT = '/kaggle/input'
WORK_ROOT = '/kaggle/working'
DS_OUT = f'{WORK_ROOT}/yolo_fusion_ds'

IMG_TR = f'{DS_OUT}/images/train'
IMG_VA = f'{DS_OUT}/images/val'
LBL_TR = f'{DS_OUT}/labels/train'
LBL_VA = f'{DS_OUT}/labels/val'
for d in (IMG_TR, IMG_VA, LBL_TR, LBL_VA):
    os.makedirs(d, exist_ok=True)

# Chuẩn ánh xạ 3 lớp giống hệt src/utils/label_mapper.py của nhóm bạn
BDD_DET_MAPPING = {
    'car': 0, 'bus': 0, 'truck': 0, 'train': 0, 'trailer': 0, 'motorcycle': 0, 'motor': 0,
    'pedestrian': 1, 'person': 1, 'rider': 1, 'bicycle': 1, 'bike': 1,
    'traffic light': 2, 'traffic_light': 2,
}
NUSCENES_DET_MAPPING = {
    'vehicle.car': 0, 'vehicle.bus.bendy': 0, 'vehicle.bus.rigid': 0,
    'vehicle.truck': 0, 'vehicle.trailer': 0, 'vehicle.construction': 0,
    'vehicle.emergency.ambulance': 0, 'vehicle.emergency.police': 0, 'vehicle.motorcycle': 0,
    'human.pedestrian.adult': 1, 'human.pedestrian.child': 1,
    'human.pedestrian.construction_worker': 1, 'human.pedestrian.police_officer': 1,
    'human.pedestrian.wheelchair': 1, 'human.pedestrian.stroller': 1,
    'human.pedestrian.personal_mobility': 1, 'vehicle.bicycle': 1,
    'traffic_light': 2,
}

def safe_symlink(src, dst):
    if not os.path.exists(dst):
        os.symlink(src, dst)

# ------------------------------------------------------------------------------
# 2.1. TÌM TRỌNG SỐ NỀN (yolo_best_kaggle_base.pt)
# ------------------------------------------------------------------------------
pt_files = sorted(glob.glob(f'{INPUT_ROOT}/**/*.pt', recursive=True))
preferred = [p for p in pt_files if any(k in os.path.basename(p).lower() for k in ['kaggle_base', 'yolo_best', 'best'])]
base_weights = preferred[0] if preferred else (pt_files[0] if pt_files else 'yolo12l.pt')
print(f'🎯 Trọng số nền khởi tạo: {base_weights}')

train_dirs, val_dirs = [], []
converted_any = False

# ------------------------------------------------------------------------------
# 2.2. AUTO-CONVERT BỘ BDD100K GỐC (JSON -> YOLO 3 LỚP + SYMLINK ẢNH)
# ------------------------------------------------------------------------------
bdd_train_jsons = glob.glob(f'{INPUT_ROOT}/**/*bdd100k_labels_images_train.json', recursive=True)
bdd_val_jsons = glob.glob(f'{INPUT_ROOT}/**/*bdd100k_labels_images_val.json', recursive=True)

def find_bdd_img_dir(split_name):
    cands = []
    for root, dirs, files in os.walk(INPUT_ROOT):
        norm = root.replace('\\', '/').lower()
        if '100k' in norm and norm.endswith(f'/{split_name}'):
            if any(f.lower().endswith('.jpg') for f in files[:20]):
                cands.append(root)
    if not cands:
        for root, dirs, files in os.walk(INPUT_ROOT):
            norm = root.replace('\\', '/').lower()
            if 'bdd' in norm and norm.endswith(f'/{split_name}') and 'label' not in norm:
                if any(f.lower().endswith('.jpg') for f in files[:20]):
                    cands.append(root)
    return cands[0] if cands else None

def convert_bdd_split(json_path, img_src_dir, img_dst_dir, lbl_dst_dir, split_tag):
    if not json_path or not img_src_dir:
        return 0
    print(f'⏳ [BDD100K-{split_tag}] Đang chuyển đổi nhãn JSON -> YOLO 3 lớp (Symlink 0MB)...')
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    avail = set(os.listdir(img_src_dir))
    count = 0
    w_img, h_img = 1280.0, 720.0
    for item in data:
        name = item.get('name', '')
        if name not in avail:
            continue
        stem = os.path.splitext(name)[0]
        lines = []
        for lbl in (item.get('labels') or []):
            box = lbl.get('box2d')
            if not box:
                continue
            cid = BDD_DET_MAPPING.get(str(lbl.get('category', '')).strip().lower())
            if cid is None:
                continue
            x1, y1, x2, y2 = float(box['x1']), float(box['y1']), float(box['x2']), float(box['y2'])
            bw, bh = (x2 - x1) / w_img, (y2 - y1) / h_img
            if bw <= 0.0 or bh <= 0.0:
                continue
            xc = max(0.0, min(1.0, (x1 + x2) / 2.0 / w_img))
            yc = max(0.0, min(1.0, (y1 + y2) / 2.0 / h_img))
            bw = max(0.0, min(1.0, bw))
            bh = max(0.0, min(1.0, bh))
            lines.append(f'{cid} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n')
        safe_symlink(os.path.join(img_src_dir, name), os.path.join(img_dst_dir, f'bdd_{name}'))
        with open(os.path.join(lbl_dst_dir, f'bdd_{stem}.txt'), 'w', encoding='utf-8') as lf:
            lf.writelines(lines)
        count += 1
    print(f'✅ [BDD100K-{split_tag}] Đã chuẩn hóa xong {count:,} ảnh!')
    return count

if bdd_train_jsons:
    n_tr = convert_bdd_split(bdd_train_jsons[0], find_bdd_img_dir('train'), IMG_TR, LBL_TR, 'TRAIN')
    if n_tr > 0:
        converted_any = True
if bdd_val_jsons:
    n_va = convert_bdd_split(bdd_val_jsons[0], find_bdd_img_dir('val'), IMG_VA, LBL_VA, 'VAL')
    if n_va > 0:
        converted_any = True

# ------------------------------------------------------------------------------
# 2.3. AUTO-CONVERT BỘ nuScenes GỐC BẰNG PURE-NUMPY (ZERO PIP INSTALL, ZERO LỖI NUMPY!)
# ------------------------------------------------------------------------------
nusc_versions = []
for root, dirs, files in os.walk(INPUT_ROOT):
    for ver in ('v1.0-mini', 'v1.0-trainval'):
        if ver in dirs and os.path.exists(os.path.join(root, ver, 'sample.json')):
            nusc_versions.append((root, ver))

if nusc_versions:
    try:
        def quat_to_rot(q):
            w, x, y, z = float(q[0]), float(q[1]), float(q[2]), float(q[3])
            return np.array([
                [1 - 2*(y*y + z*z), 2*(x*y - z*w),     2*(x*z + y*w)],
                [2*(x*y + z*w),     1 - 2*(x*x + z*z), 2*(y*z - x*w)],
                [2*(x*z - y*w),     2*(y*z + x*w),     1 - 2*(x*x + y*y)],
            ], dtype=np.float64)

        nusc_root, nusc_ver = nusc_versions[0]
        meta_dir = os.path.join(nusc_root, nusc_ver)
        print(f'⏳ [nuScenes {nusc_ver}] Pure-NumPy trích xuất CAM_FRONT -> YOLO 3 lớp (Symlink 0MB)...')

        def load_tbl(fname):
            with open(os.path.join(meta_dir, fname), 'r', encoding='utf-8') as f:
                return json.load(f)

        samples = load_tbl('sample.json')
        sdata_by_tok = {x['token']: x for x in load_tbl('sample_data.json')}
        ego_by_tok = {x['token']: x for x in load_tbl('ego_pose.json')}
        cs_by_tok = {x['token']: x for x in load_tbl('calibrated_sensor.json')}
        sensor_by_tok = {x['token']: x for x in load_tbl('sensor.json')}
        cat_by_tok = {x['token']: x['name'] for x in load_tbl('category.json')}
        inst_to_cat = {x['token']: cat_by_tok.get(x['category_token'], '') for x in load_tbl('instance.json')}

        anns_by_sample = {}
        for ann in load_tbl('sample_annotation.json'):
            anns_by_sample.setdefault(ann['sample_token'], []).append(ann)

        cam_front_by_sample = {}
        for sd in sdata_by_tok.values():
            if not sd.get('is_key_frame'):
                continue
            cs = cs_by_tok.get(sd['calibrated_sensor_token'])
            if not cs:
                continue
            sens = sensor_by_tok.get(cs['sensor_token'])
            if sens and sens.get('channel') == 'CAM_FRONT':
                cam_front_by_sample[sd['sample_token']] = (sd, cs)

        unit_corners = np.array([
            [ 0.5,  0.5,  0.5], [ 0.5,  0.5, -0.5], [ 0.5, -0.5,  0.5], [ 0.5, -0.5, -0.5],
            [-0.5,  0.5,  0.5], [-0.5,  0.5, -0.5], [-0.5, -0.5,  0.5], [-0.5, -0.5, -0.5],
        ], dtype=np.float64).T

        n_nusc_tr, n_nusc_va = 0, 0
        for idx_s, sample in enumerate(samples):
            stok = sample['token']
            if stok not in cam_front_by_sample:
                continue
            sd, cs = cam_front_by_sample[stok]
            raw_p = os.path.join(nusc_root, sd['filename'])
            if not os.path.exists(raw_p):
                continue
            im_w, im_h = float(sd.get('width', 1600)), float(sd.get('height', 900))
            ego = ego_by_tok[sd['ego_pose_token']]
            R_ego = quat_to_rot(ego['rotation'])
            t_ego = np.array(ego['translation'], dtype=np.float64).reshape(3, 1)
            R_cs = quat_to_rot(cs['rotation'])
            t_cs = np.array(cs['translation'], dtype=np.float64).reshape(3, 1)
            K = np.array(cs['camera_intrinsic'], dtype=np.float64)

            lines = []
            for ann in anns_by_sample.get(stok, []):
                cname = inst_to_cat.get(ann['instance_token'], '').strip().lower()
                cid = NUSCENES_DET_MAPPING.get(cname)
                if cid not in (0, 1):
                    continue
                w_b, l_b, h_b = ann['size']
                dims = np.array([[l_b], [w_b], [h_b]], dtype=np.float64)
                R_b = quat_to_rot(ann['rotation'])
                t_b = np.array(ann['translation'], dtype=np.float64).reshape(3, 1)
                pts_global = R_b @ (unit_corners * dims) + t_b
                pts_ego = R_ego.T @ (pts_global - t_ego)
                pts_cam = R_cs.T @ (pts_ego - t_cs)
                if np.all(pts_cam[2, :] <= 0.5):
                    continue
                valid = pts_cam[:, pts_cam[2, :] > 0.5]
                proj = K @ valid
                u = proj[0, :] / proj[2, :]
                v = proj[1, :] / proj[2, :]
                x1, y1 = max(0.0, float(u.min())), max(0.0, float(v.min()))
                x2, y2 = min(im_w, float(u.max())), min(im_h, float(v.max()))
                bw_px, bh_px = x2 - x1, y2 - y1
                if bw_px <= 15 or bh_px <= 15 or x1 >= im_w or y1 >= im_h or x2 <= 0 or y2 <= 0:
                    continue
                xc = max(0.0, min(1.0, (x1 + bw_px / 2.0) / im_w))
                yc = max(0.0, min(1.0, (y1 + bh_px / 2.0) / im_h))
                lines.append(f'{cid} {xc:.6f} {yc:.6f} {bw_px/im_w:.6f} {bh_px/im_h:.6f}\n')

            is_val_s = (idx_s % 7 == 0)
            dst_i = IMG_VA if is_val_s else IMG_TR
            dst_l = LBL_VA if is_val_s else LBL_TR
            fname_cam = os.path.basename(sd['filename'])
            bname = f'nusc_{fname_cam}'
            safe_symlink(raw_p, os.path.join(dst_i, bname))
            with open(os.path.join(dst_l, f'{os.path.splitext(bname)[0]}.txt'), 'w', encoding='utf-8') as lf:
                lf.writelines(lines)
            if is_val_s:
                n_nusc_va += 1
            else:
                n_nusc_tr += 1
        print(f'✅ [nuScenes Pure-NumPy] Đã trích xuất {n_nusc_tr:,} ảnh Train + {n_nusc_va:,} ảnh Val!')
        converted_any = True
    except Exception as e:
        print(f'⚠️ [nuScenes Warning] Bỏ qua nuScenes ({e}), tiếp tục train Full BDD100K!')

if converted_any:
    if len(os.listdir(IMG_TR)) > 0:
        train_dirs.append(IMG_TR)
    if len(os.listdir(IMG_VA)) > 0:
        val_dirs.append(IMG_VA)

# ------------------------------------------------------------------------------
# 2.4. QUÉT THÊM CÁC BỘ YOLO .TXT ĐÃ CHUẨN HÓA SẴN (NẾU CÓ)
# ------------------------------------------------------------------------------
def has_yolo_labels(img_dir):
    cand = img_dir.replace('/images/', '/labels/').replace('/images', '/labels')
    return os.path.isdir(cand) and any(f.endswith('.txt') for f in os.listdir(cand)[:20])

for root, dirs, files in os.walk(INPUT_ROOT):
    norm = root.replace('\\', '/').lower()
    if '/labels' in norm:
        continue
    if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files[:20]) and has_yolo_labels(root):
        if (norm.endswith('/train') or '/train/' in norm) and root not in train_dirs:
            train_dirs.append(root)
        elif (norm.endswith('/val') or norm.endswith('/valid') or '/val/' in norm) and root not in val_dirs:
            val_dirs.append(root)

assert len(train_dirs) > 0 and len(val_dirs) > 0, '❌ Chưa tìm thấy dataset BDD100K/nuScenes trong /kaggle/input! Hãy bấm Add Input.'

tot_tr = sum(len(os.listdir(d)) for d in train_dirs)
tot_va = sum(len(os.listdir(d)) for d in val_dirs)
print(f'\n🚀 SẴN SÀNG FUSION DATASET: {tot_tr:,} ảnh Train | {tot_va:,} ảnh Val')

YAML_PATH = f'{WORK_ROOT}/data_kaggle_fusion.yaml'
with open(YAML_PATH, 'w', encoding='utf-8') as f:
    yaml.safe_dump({
        'train': train_dirs if len(train_dirs) > 1 else train_dirs[0],
        'val': val_dirs if len(val_dirs) > 1 else val_dirs[0],
        'nc': 3,
        'names': {0: 'Vehicle', 1: 'Pedestrian', 2: 'Traffic Light'},
    }, f, sort_keys=False, allow_unicode=True)
print(f'✅ Đã xuất cấu hình YAML: {YAML_PATH}')

In [ ]:
# ==============================================================================
# BƯỚC 3: FINE-TUNE MAX-PING YOLOv12-LARGE TRÊN GPU T4 x 2 (CHẠY NGẦM QUA ĐÊM)
# ==============================================================================
RUN_PROJECT = f'{WORK_ROOT}/runs'
RUN_NAME = 'yolo12l_joint_bdd_nuscenes_maxping'

model = YOLO(base_weights)

print('\n🔥 BẮT ĐẦU FINE-TUNE MAX-PING YOLOv12-LARGE (imgsz=800, time=10.5h)...')
results = model.train(
    data=YAML_PATH,
    epochs=15,
    time=10.5,              # Tự dừng êm ở 10.5h (chừa dư 1.5h cho Bước 4 Val + TTA trước mốc 12h Kaggle!)
    imgsz=800,              # Giữ độ phân giải 800px sắc nét cho Traffic Light & Pedestrian ở xa
    batch=BATCH_SIZE,       # 8 trên T4 x2 (4 ảnh/GPU - 100% chống OOM), 4 trên 1 GPU
    nbs=64,                 # Gradient accumulation chuẩn 64 giúp hội tụ cực mượt
    device=DEVICES,
    patience=6,
    optimizer='AdamW',
    lr0=0.00012,            # Learning rate tinh chỉnh mượt từ bản 72.43% (không phá trọng số cũ)
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=1.5,
    weight_decay=0.0005,
    # Bộ Loss & Augmentation tối ưu cho 3 lớp ADAS (Vehicle, Pedestrian, Traffic Light)
    box=7.5,
    cls=0.6,
    dfl=1.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.8,
    mixup=0.10,
    copy_paste=0.10,
    close_mosaic=4,         # Tắt Mosaic 4 epoch cuối khóa khít viền BBox (+2% mAP50-95)
    amp=True,
    workers=4,
    cache=False,            # An toàn ổ cứng 20GB của Kaggle
    project=RUN_PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    save=True,
    plots=True
)
print('🎉 ĐÃ HOÀN TẤT HUẤN LUYỆN!')

In [ ]:
# ==============================================================================
# BƯỚC 4: ĐÁNH GIÁ MAX-PING TTA (augment=True) & ĐÓNG GÓI OUTPUT SẠCH ~51 MB
# ==============================================================================
from ultralytics.utils.torch_utils import strip_optimizer

run_dir = f'{RUN_PROJECT}/{RUN_NAME}'
best_pt = f'{run_dir}/weights/best.pt'
export_dir = f'{WORK_ROOT}/FINAL_EXPORT'
os.makedirs(export_dir, exist_ok=True)

if os.path.exists(best_pt):
    final_pt = f'{export_dir}/yolo_best.pt'
    shutil.copy2(best_pt, final_pt)
    strip_optimizer(final_pt)

    for fname in os.listdir(run_dir):
        if fname.endswith(('.csv', '.png', '.jpg', '.yaml')):
            shutil.copy2(os.path.join(run_dir, fname), os.path.join(export_dir, fname))

    eval_model = YOLO(final_pt)
    m_std = eval_model.val(data=YAML_PATH, imgsz=800, batch=8, half=True, split='val', augment=False, device=0)
    m_tta = eval_model.val(data=YAML_PATH, imgsz=800, batch=8, half=True, split='val', augment=True, conf=0.001, iou=0.65, device=0)

    summary_txt = (
        '==============================================================\n'
        '📊 KẾT QUẢ CHÍNH THỨC TRÊN TẬP VAL (STANDARD imgsz=800):\n'
        f'  • Precision : {m_std.box.mp * 100:.2f}%\n'
        f'  • Recall    : {m_std.box.mr * 100:.2f}%\n'
        f'  • mAP@50    : {m_std.box.map50 * 100:.2f}%\n'
        f'  • mAP@50-95 : {m_std.box.map * 100:.2f}%\n'
        '--------------------------------------------------------------\n'
        '🏆 KẾT QUẢ MAX-PING VỚI TTA (augment=True, imgsz=800):\n'
        f'  • Precision : {m_tta.box.mp * 100:.2f}%\n'
        f'  • Recall    : {m_tta.box.mr * 100:.2f}%\n'
        f'  • mAP@50    : {m_tta.box.map50 * 100:.2f}%\n'
        f'  • mAP@50-95 : {m_tta.box.map * 100:.2f}%\n'
        f'  • File size : {os.path.getsize(final_pt)/(1024**2):.2f} MB ({final_pt})\n'
        '==============================================================\n'
    )
    print(summary_txt)
    with open(f'{export_dir}/FINAL_METRICS_SUMMARY.txt', 'w', encoding='utf-8') as f:
        f.write(summary_txt)

    shutil.rmtree(DS_OUT, ignore_errors=True)
    print(f'✅ Toàn bộ file trọng số + biểu đồ + CSV đã sẵn sàng tại: {export_dir}')
else:
    print('⚠️ Không tìm thấy best.pt! Kiểm tra lại log huấn luyện ở Bước 3.')